In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# ==========================================
# 0. Path Definitions & Style
# ==========================================
BASE_DIR = Path('/kyb/agpd/wwang/data/')
OUT_DIR = Path('/kyb/agpd/wwang/sst-pomdp/outputs/')

# Journal Style Settings
sns.set_context("paper", font_scale=1.4)
sns.set_style("white") # Clean background (no grid)

# Display Names for Titles (LaTeX for params, clean text for clinical)
DISPLAY_NAMES = {
    # POMDP Parameters
    'q_d_n': r"$\chi'$",
    'q_d': r"$\chi$",
    'q_s_n': r"$\sigma'$",
    'q_s': r"$\sigma$",
    'cost_stop_error': r"$c_{\mathrm{se}}$",
    'cost_time': r"$c_t$",
    # Clinical / Behavioral
    'adhd': 'ADHD Score',
    'mrt_gs': 'GSRT',
    'ssrt': 'SSRT',
}

# ==========================================
# 1. Data Loading & Filtering
# ==========================================
def load_and_filter_data():
    print(f"Loading data from {OUT_DIR}...")
    
    # Load Main Files
    df_emb = pd.read_csv(OUT_DIR / 'embeddings.csv')
    df_stats = pd.read_csv(OUT_DIR / 'stats_beh_full.csv')
    df_post = pd.read_csv(OUT_DIR / 'posterior_summary.csv')
    

    # --- Prepare Main Data ---
    df_emb['year'] = df_emb['year'].astype(str)
    df_stats['year'] = df_stats['year'].astype(str)
    df_post['subject_year'] = df_post['subject_year'].astype(str)

    # Pivot Posterior
    df_post_pivot = df_post.pivot_table(index=['subject_id', 'subject_year'], 
                                        columns='param', 
                                        values='mean').reset_index()
    df_post_pivot.rename(columns={'subject_year': 'year'}, inplace=True)

    # Merge Sequence
    df_merged_1 = pd.merge(df_emb, df_stats, on=['subject_id', 'year', 'filename'], how='inner')
    df_final = pd.merge(df_merged_1, df_post_pivot, on=['subject_id', 'year'], how='inner')
        
    print(f"Data Loaded. Valid Subjects: {len(df_final)}")
    return df_final

# ==========================================
# 3. Visualization: PCA Parameters (Plot 1)
# ==========================================
def plot_pca_param(df):
    print("Generating PCA...")

    embedding_cols = [f'emb_{i}' for i in range(64)]
    X = df[embedding_cols].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Run PCA
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(X_scaled)
    df['pca_1'] = pca_result[:, 0]
    df['pca_2'] = pca_result[:, 1]
    
    # Define Targets: 6 Parameters + 3 Clinical
    targets = [
        'q_d_n', 'q_d', 'q_s_n', 
        'q_s', 'cost_stop_error', 'cost_time',
        'adhd', 'mrt_gs', 'ssrt'
    ]
    
    # Calculate Variance Ratios
    var_ratios = pca.explained_variance_ratio_ * 100
    pc1_label = f"PC 1 ({var_ratios[0]:.1f}%)"
    pc2_label = f"PC 2 ({var_ratios[1]:.1f}%)"
    
    # Create 3x3 Grid
    # sharex/sharey helps alignment but can hide ticks if data ranges differ significantly.
    # Since all plots are on the same PCA coords (pca_1, pca_2), sharing is safe.
    fig, axes = plt.subplots(3, 3, figsize=(12, 11), sharex=True, sharey=True)
    axes = axes.flatten()

    for i, var in enumerate(targets):
        if var in df.columns:
            # Scatter Plot
            # Using 'mako_r' for parameters to look scientific/cool
            # You can switch colormap for clinical vars if you want distinct visual styles
            cmap_choice = 'RdYlBu_r'
            
            vmin, vmax = df[var].quantile([0.01, 0.99]) # Robust scaling for color
            
            sc = axes[i].scatter(df['pca_1'], df['pca_2'], 
                                 c=df[var], cmap=cmap_choice, 
                                 alpha=0.7, s=8, linewidth=0, 
                                 vmin=vmin, vmax=vmax)
            
            # Title
            title = DISPLAY_NAMES.get(var, var)
            axes[i].set_title(title, fontsize=14)
            
            # Clean Spines (L-Shape)
            axes[i].spines['top'].set_visible(False)
            axes[i].spines['right'].set_visible(False)
            
            # Labels only on bottom row and left column
            if i >= 6: # Bottom row indices are 6, 7, 8
                axes[i].set_xlabel(pc1_label, fontsize=12)
            if i % 3 == 0: # Left column indices are 0, 3, 6
                axes[i].set_ylabel(pc2_label, fontsize=12)
            
            # Colorbar
            cbar = plt.colorbar(sc, ax=axes[i], pad=0.02)
            cbar.ax.tick_params(labelsize=9)
            cbar.outline.set_visible(False)
        else:
            axes[i].text(0.5, 0.5, f"{var} missing", ha='center')
            axes[i].axis('off')
            
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'embed_pca.png', dpi=300, bbox_inches='tight')
    plt.show()

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    # 1. Load
    df = load_and_filter_data()
    
    # 2. Plot
    plot_pca_param(df)

Loading data from /kyb/agpd/wwang/sst-pomdp/outputs...
Data Loaded. Valid Subjects: 5918
Generating PCA...


In [9]:
import plotly.graph_objects as go
from plotly.offline import plot
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ... (Previous imports and data loading code remains the same) ...

def plot_pca_3d_clean(df, filename="interactive_pca_3d_clean.html"):
    print("Generating Clean Interactive 3D PCA...")
    
    # 1. Run PCA
    embedding_cols = [f'emb_{i}' for i in range(64)]
    X = StandardScaler().fit_transform(df[embedding_cols].values)
    
    pca = PCA(n_components=3)
    res = pca.fit_transform(X)
    
    df['pca_1'] = res[:, 0]
    df['pca_2'] = res[:, 1]
    df['pca_3'] = res[:, 2]
    
    var_exp = pca.explained_variance_ratio_
    total_var = sum(var_exp) * 100
    print(f"Explained Variance: {total_var:.2f}%")

    targets = [
        'adhd', 'mrt_gs', 'ssrt', 
        'q_d', 'q_d_n', 'q_s', 'q_s_n', 
        'cost_stop_error', 'cost_time'
    ]

    fig = go.Figure()

    # Default Trace
    default_var = targets[0]
    fig.add_trace(go.Scatter3d(
        x=df['pca_1'], y=df['pca_2'], z=df['pca_3'],
        mode='markers',
        marker=dict(
            size=4, # Slightly larger dots
            color=df[default_var],
            colorscale='Spectral_r', # Try Spectral for more visible gradient than RdYlBu
            opacity=0.9,
            colorbar=dict(title=DISPLAY_NAMES.get(default_var, default_var), len=0.5)
        ),
        text=[f"ID: {s}<br>{default_var}: {v:.2f}" for s, v in zip(df['subject_id'], df[default_var])],
        hoverinfo='text'
    ))

    # Dropdown Logic
    buttons = []
    for var in targets:
        if var not in df.columns: continue
        label = DISPLAY_NAMES.get(var, var)
        
        # Clinical = Spectral (Red-Yellow-Blue-ish), Params = Viridis or Plasma (Scientific)
        # 'Spectral_r' usually shows gradients better than RdYlBu_r for 3D clouds
        cmap = 'Spectral_r' if var in ['adhd', 'mrt_gs', 'ssrt'] else 'Viridis' 
        
        buttons.append(dict(
            label=label,
            method="restyle",
            args=[{
                'marker.color': [df[var]],
                'marker.colorscale': cmap,
                'marker.colorbar.title': label,
                'text': [[f"ID: {s}<br>{label}: {v:.2f}" for s, v in zip(df['subject_id'], df[var])]]
            }]
        ))

    # Clean Layout (No Grids, No Background)
    axis_template = dict(
        showgrid=False,      # Remove grid lines
        zeroline=False,      # Remove zero line
        showbackground=False,# Remove background color
        showticklabels=False,# Optional: remove numbers if you want pure art
        title=''             # Hide axis titles for cleaner look (optional)
    )

    fig.update_layout(
        title=f"Latent Policy Space (PCA Var: {total_var:.1f}%)",
        width=1200,  # Bigger
        height=900,
        scene=dict(
            xaxis=axis_template,
            yaxis=axis_template,
            zaxis=axis_template,
            aspectmode='cube' # Keep proportions balanced
        ),
        updatemenus=[dict(
            active=0, buttons=buttons,
            x=0.05, y=0.95, xanchor='left', yanchor='top',
            bgcolor='rgba(255,255,255,0.8)'
        )],
        paper_bgcolor='white'
    )

    save_path = OUT_DIR / filename
    plot(fig, filename=str(save_path), auto_open=False)
    print(f"Saved clean 3D plot to: {save_path}")

# 使用方式：
if __name__ == "__main__":
    df = load_and_filter_data() # 假设你上面的函数还在
    plot_pca_3d_clean(df)

Loading data from /kyb/agpd/wwang/sst-pomdp/outputs...
Data Loaded. Valid Subjects: 5918
Generating Clean Interactive 3D PCA...
Explained Variance: 76.70%
Saved clean 3D plot to: /kyb/agpd/wwang/sst-pomdp/outputs/interactive_pca_3d_clean.html
